# Dysgraphia Image Training (Google Colab)

This notebook trains a handwriting image classifier using transfer learning (ConvNeXtV2) on your dysgraphia dataset.

Dataset source is downloaded directly from Kaggle:
- https://www.kaggle.com/datasets/bhdivyansh/dysgraphia-dataset

Expected extracted structure:
- `Dysgraphia-dataset/PD`
- `Dysgraphia-dataset/LPD`
- `Dysgraphia-dataset/image_sentences.csv` (optional metadata)

In [ ]:
!nvidia-smi || true
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip -q install timm==1.0.25 scikit-learn==1.3.2 pyyaml pillow kaggle

In [ ]:
from google.colab import files
import os

print('Upload your kaggle.json API token file (from Kaggle Account > Create New API Token).')
uploaded = files.upload()

if 'kaggle.json' not in uploaded:
    raise FileNotFoundError('kaggle.json was not uploaded.')

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle API credentials configured.')

In [ ]:
import os
import zipfile
from pathlib import Path

KAGGLE_DATASET = 'bhdivyansh/dysgraphia-dataset'
WORK_ROOT = Path('/content/dysgraphia_image')
RAW_ROOT = WORK_ROOT / 'data' / 'raw'
SOURCE_DATASET = RAW_ROOT / 'source_dataset'
ARTIFACTS = WORK_ROOT / 'artifacts'
KAGGLE_TMP = Path('/content/kaggle_tmp')

for p in [SOURCE_DATASET, ARTIFACTS / 'checkpoints', ARTIFACTS / 'exported', ARTIFACTS / 'reports', KAGGLE_TMP]:
    p.mkdir(parents=True, exist_ok=True)

print('Working root:', WORK_ROOT)

In [ ]:
import shutil
import subprocess

def download_from_kaggle(dataset_ref: str, destination: Path):
    destination.mkdir(parents=True, exist_ok=True)
    zip_path = destination / 'dataset.zip'

    cmd = [
        'kaggle',
        'datasets',
        'download',
        '-d',
        dataset_ref,
        '-p',
        str(destination),
        '--force'
    ]
    subprocess.run(cmd, check=True)

    if not zip_path.exists():
        candidates = list(destination.glob('*.zip'))
        if not candidates:
            raise FileNotFoundError('Kaggle dataset zip not found after download.')
        zip_path = candidates[0]

    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(destination)

    return zip_path

def copy_dataset(src: Path, dst: Path):
    if dst.exists():
        for item in dst.iterdir():
            if item.is_dir():
                shutil.rmtree(item)
            else:
                item.unlink()
    dst.mkdir(parents=True, exist_ok=True)

    for item in src.iterdir():
        target = dst / item.name
        if item.is_dir():
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)

zip_file = download_from_kaggle(KAGGLE_DATASET, KAGGLE_TMP)

# Kaggle archive may extract into Dysgraphia-dataset/ or directly into root.
nested = KAGGLE_TMP / 'Dysgraphia-dataset'
dataset_root = nested if nested.exists() else KAGGLE_TMP

copy_dataset(dataset_root, SOURCE_DATASET)
print('Downloaded zip:', zip_file)
print('Copied dataset to', SOURCE_DATASET)
print('Top-level contents:', [p.name for p in SOURCE_DATASET.iterdir()])

In [ ]:
import random

CLASS_MAP = {'PD': 'dysgraphia', 'LPD': 'typical'}

def list_images(folder: Path):
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    return [p for p in folder.rglob('*') if p.suffix.lower() in exts]

def prepare_splits(source_root: Path, raw_root: Path, seed=42, train_ratio=0.7, val_ratio=0.15):
    random.seed(seed)
    split_root = raw_root

    for split in ['train', 'val', 'test']:
        for cls in CLASS_MAP.values():
            dst = split_root / split / cls
            dst.mkdir(parents=True, exist_ok=True)
            for f in dst.glob('*'):
                if f.is_file():
                    f.unlink()

    summary = {}
    for src_cls, dst_cls in CLASS_MAP.items():
        files = list_images(source_root / src_cls)
        if not files:
            raise ValueError(f'No images found in {source_root / src_cls}')

        random.shuffle(files)
        n = len(files)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)

        train_files = files[:n_train]
        val_files = files[n_train:n_train+n_val]
        test_files = files[n_train+n_val:]

        for split, split_files in [('train', train_files), ('val', val_files), ('test', test_files)]:
            dst_dir = split_root / split / dst_cls
            for src in split_files:
                dst = dst_dir / src.name
                if dst.exists():
                    dst = dst_dir / f'{src.stem}_{random.randint(1, 999999)}{src.suffix}'
                shutil.copy2(src, dst)

        summary[f'{dst_cls}_total'] = n
        summary[f'{dst_cls}_train'] = len(train_files)
        summary[f'{dst_cls}_val'] = len(val_files)
        summary[f'{dst_cls}_test'] = len(test_files)

    return summary

summary = prepare_splits(SOURCE_DATASET, RAW_ROOT, seed=42)
summary

In [ ]:
import json
import timm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix, f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 12
LR = 1e-4
WEIGHT_DECAY = 1e-4
MODEL_NAME = 'convnextv2_base'

mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE[0] + 24, IMG_SIZE[1] + 24)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(degrees=6),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

eval_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_ds = datasets.ImageFolder(str(RAW_ROOT / 'train'), transform=train_tfms)
val_ds = datasets.ImageFolder(str(RAW_ROOT / 'val'), transform=eval_tfms)
test_ds = datasets.ImageFolder(str(RAW_ROOT / 'test'), transform=eval_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print('Class mapping:', train_ds.class_to_idx)

model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=2).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

In [ ]:
def run_epoch(loader, training=True):
    model.train(training)
    total_loss = 0.0
    labels_all = []
    preds_all = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.set_grad_enabled(training):
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(images)
                loss = loss_fn(logits, labels)

            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)
        labels_all.extend(labels.detach().cpu().tolist())
        preds_all.extend(preds.detach().cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    f1 = f1_score(labels_all, preds_all, average='macro')
    acc = (torch.tensor(labels_all) == torch.tensor(preds_all)).float().mean().item()
    return {'loss': float(avg_loss), 'f1_macro': float(f1), 'accuracy': float(acc)}

best_val_f1 = -1.0
best_path = ARTIFACTS / 'checkpoints' / 'best_convnextv2_dysgraphia.pt'
history = []

for epoch in range(1, EPOCHS + 1):
    train_m = run_epoch(train_loader, training=True)
    val_m = run_epoch(val_loader, training=False)
    scheduler.step()

    row = {'epoch': epoch, 'train': train_m, 'val': val_m}
    history.append(row)
    print(f"Epoch {epoch}/{EPOCHS} | train_loss={train_m['loss']:.4f} train_f1={train_m['f1_macro']:.4f} | val_loss={val_m['loss']:.4f} val_f1={val_m['f1_macro']:.4f}")

    if val_m['f1_macro'] > best_val_f1:
        best_val_f1 = val_m['f1_macro']
        torch.save({
            'model_state_dict': model.state_dict(),
            'class_to_idx': train_ds.class_to_idx,
            'model_name': MODEL_NAME,
            'image_size': IMG_SIZE,
        }, best_path)

(ARTIFACTS / 'reports').mkdir(parents=True, exist_ok=True)
with open(ARTIFACTS / 'reports' / 'train_history.json', 'w', encoding='utf-8') as f:
    json.dump(history, f, indent=2)

print('Best val F1:', best_val_f1)
print('Best checkpoint:', best_path)

In [ ]:
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

all_labels = []
all_preds = []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        preds = torch.argmax(logits, dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.tolist())

idx_to_class = {v: k for k, v in train_ds.class_to_idx.items()}
target_names = [idx_to_class[i] for i in sorted(idx_to_class.keys())]
report = classification_report(all_labels, all_preds, target_names=target_names, output_dict=True, zero_division=0)
conf = confusion_matrix(all_labels, all_preds).tolist()

with open(ARTIFACTS / 'reports' / 'test_classification_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)
with open(ARTIFACTS / 'reports' / 'test_confusion_matrix.json', 'w', encoding='utf-8') as f:
    json.dump(conf, f, indent=2)

exported_path = ARTIFACTS / 'exported' / 'dysgraphia_image_model.pt'
torch.save(ckpt, exported_path)

print('Test accuracy:', report['accuracy'])
print('Test F1 macro:', report['macro avg']['f1-score'])
print('Exported model:', exported_path)

In [ ]:
# Optional: package artifacts as a zip for download
import shutil
from google.colab import files

zip_base = '/content/dysgraphia_image_artifacts'
zip_path = shutil.make_archive(zip_base, 'zip', ARTIFACTS)
print('Created:', zip_path)
files.download(zip_path)

# If you still want Drive copy, mount Drive manually and then copy ZIP file.